# 03 · Time evolution with TEBD and TDVP

We will evolve a four-spin state after a change of magnetic field. Two complete algorithms are implemented:

- second-order TEBD, which applies local two-site gates and can enlarge the MPS bond dimension,
- symmetric one-site TDVP, which evolves center tensors forward and bond tensors backward while preserving the chosen bond dimensions.

The notebook accompanies **Time evolving block decimation**, **Canonical forms**, and **Time-dependent variational principle** in the [lecture notes](https://arxiv.org/abs/2512.02706). It is self-contained. Use Python 3.10 or later and NumPy 1.26 or later, then run the cells in order. No tensor-network library is used.

The chain is small enough to compare both calculations with exact unitary evolution. Dense matrices also let us check the finite-MPS tangent projector directly. The TDVP integrator itself uses local MPO environments, rather than the dense many-body Hamiltonian or the dense tangent projector.


In [1]:
import sys
import time
import numpy as np

assert sys.version_info >= (3, 10)
assert tuple(map(int, np.__version__.split(".")[:2])) >= (1, 26)
np.set_printoptions(precision=6, suppress=True)
started = time.perf_counter()
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.diag([1, -1]).astype(complex)
N, J, hx, hz = 4, 1.0, 1.2, 0.1
print("NumPy", np.__version__)


NumPy 2.5.3


## Hamiltonian, state ordering, and the initial state

We use the same Pauli-matrix convention as the MPO and DMRG notebook:
$$
H=-J\sum_{i=0}^{N-2}Z_iZ_{i+1}-h_x\sum_iX_i-h_z\sum_iZ_i.
$$
Site zero is the leftmost Kronecker factor. An MPS tensor is `A[a,s,b]` with shape $(D_i,2,D_{i+1})$. Its physical leg points downward in the diagrams. An MPO tensor is `W[mu,nu,s,t]`, where $s$ is the bra/output index and $t$ the ket/input index.

The initial state will approximate the ground state at $h_x=0.4$, $h_z=0.2$, with maximum bond dimension two. We then apply a local phase rotation before evolving with the fields specified above. The rotation preserves every Schmidt rank and makes the initial tensors complex, so the calculation tests conjugation conventions as well as real arithmetic.


In [2]:
def operator_on_sites(N, insertions):
    result = np.ones((1, 1), dtype=complex)
    for site in range(N):
        result = np.kron(result, insertions.get(site, I))
    return result


def dense_ising(N, J, hx, hz):
    H = np.zeros((2**N, 2**N), dtype=complex)
    for i in range(N):
        H -= hx * operator_on_sites(N, {i: X})
        H -= hz * operator_on_sites(N, {i: Z})
    for i in range(N - 1):
        H -= J * operator_on_sites(N, {i: Z, i + 1: Z})
    return H


def mps_to_dense(mps):
    state = np.ones((1, 1), dtype=complex)
    for A in mps:
        state = np.einsum("xa,asb->xsb", state, A).reshape(-1, A.shape[2])
    return state[:, 0]

H_dense = dense_ising(N, J, hx, hz)
_, initial_vectors = np.linalg.eigh(dense_ising(N, J, 0.4, 0.2))
initial_dense = initial_vectors[:, 0]


## Move the canonical center without changing the state

We need two factorizations. Reshaping a site as $A_{(a,s),b}$ and using QR gives $A=QC$. The columns of $Q$ are orthonormal, so $Q$ is left-canonical and $C$ can be absorbed into the next tensor.

For motion to the left, reshape as $A_{a,(s,b)}$ and apply QR to $A^\dagger$. If $A^\dagger=QR$, then $A=R^\dagger Q^\dagger$. Store $C=R^\dagger$ and the right-canonical tensor $Q^\dagger$. This conjugate transpose matters for complex states.

`move_center` applies these exact factorizations on both sides of the requested site. It is intentionally explicit and rebuilds the canonical form each time. A large-chain implementation would retain the current center position and move it only as far as necessary.


In [3]:
def factor_left(A):
    Dl, d, Dr = A.shape
    Q, C = np.linalg.qr(A.reshape(Dl * d, Dr), mode="reduced")
    return Q.reshape(Dl, d, -1), C


def factor_right(A):
    Dl, d, Dr = A.shape
    Q, R = np.linalg.qr(A.reshape(Dl, d * Dr).conj().T, mode="reduced")
    return R.conj().T, Q.conj().T.reshape(-1, d, Dr)


def move_center(mps, site):
    for i in range(site):
        mps[i], C = factor_left(mps[i])
        mps[i + 1] = np.einsum("ab,bsd->asd", C, mps[i + 1])
    for i in range(len(mps) - 1, site, -1):
        C, mps[i] = factor_right(mps[i])
        mps[i - 1] = np.einsum("asb,bc->asc", mps[i - 1], C)


def mps_from_dense(state, N, chi):
    remaining, Dl, mps = state.reshape(1, -1), 1, []
    for _ in range(N - 1):
        U, singular, Vh = np.linalg.svd(remaining.reshape(Dl * 2, -1),
                                       full_matrices=False)
        keep = min(chi, len(singular))
        mps.append(U[:, :keep].reshape(Dl, 2, keep))
        remaining = singular[:keep, None] * Vh[:keep]
        Dl = keep
    mps.append(remaining.reshape(Dl, 2, 1))
    mps[-1] /= np.linalg.norm(mps_to_dense(mps))
    move_center(mps, 0)
    return mps

mps_initial = mps_from_dense(initial_dense, N, chi=2)
mps_initial[1][:, 1, :] *= np.exp(0.23j)
psi_initial = mps_to_dense(mps_initial)
move_center(mps_initial, 2)
assert np.allclose(mps_to_dense(mps_initial), psi_initial, atol=1e-12)
move_center(mps_initial, 0)
assert np.allclose(mps_to_dense(mps_initial), psi_initial, atol=1e-12)
assert abs(np.linalg.norm(psi_initial) - 1) < 1e-12
E_initial = float(np.vdot(psi_initial, H_dense @ psi_initial).real)
print("Initial MPS shapes:", [A.shape for A in mps_initial])
print(f"Energy under the post-quench Hamiltonian: {E_initial:.12f}")


Initial MPS shapes: [(1, 2, 2), (2, 2, 2), (2, 2, 2), (2, 2, 1)]
Energy under the post-quench Hamiltonian: -4.445416067264


## Split the Hamiltonian into bond terms

For open boundaries, an end site belongs to one bond and an interior site to two. Set $q_i=1$ at an end and $q_i=2$ in the interior. With $f=-h_xX-h_zZ$, choose
$$
h_{i,i+1}=-JZ\otimes Z+\frac{f\otimes I}{q_i}
                         +\frac{I\otimes f}{q_{i+1}}.
$$
Then $H=\sum_i h_{i,i+1}$, including the endpoint fields with their full strength. We check this identity against the independent Pauli-string Hamiltonian.

`unitary_action` uses an eigendecomposition of a Hermitian matrix to apply $e^{-iH\delta t}$ to a tensor. Passing a negative time interval will give the backward evolution needed for TDVP bond tensors later.


In [4]:
def bond_hamiltonians(N, J, hx, hz):
    degree = np.full(N, 2)
    degree[[0, -1]] = 1
    field = -hx * X - hz * Z
    return [-J * np.kron(Z, Z) + np.kron(field, I) / degree[i]
            + np.kron(I, field) / degree[i + 1] for i in range(N - 1)]


def unitary_action(matrix, tensor, interval):
    assert np.linalg.norm(matrix - matrix.conj().T) < 1e-10
    energies, vectors = np.linalg.eigh(matrix)
    coefficients = vectors.conj().T @ tensor.ravel()
    evolved = vectors @ (np.exp(-1j * interval * energies) * coefficients)
    return evolved.reshape(tensor.shape)


def gate_matrix(h, interval):
    energies, vectors = np.linalg.eigh(h)
    return (vectors * np.exp(-1j * interval * energies)) @ vectors.conj().T

bonds = bond_hamiltonians(N, J, hx, hz)
assembled = np.zeros_like(H_dense)
for i, h in enumerate(bonds):
    assembled += np.kron(np.kron(np.eye(2**i), h), np.eye(2**(N - i - 2)))
assert np.allclose(assembled, H_dense, atol=1e-12)
print("Bond decomposition error:", np.linalg.norm(assembled - H_dense))


Bond decomposition error: 8.15843973306311e-16


## TEBD: apply a gate to a canonical pair

Bring the center to the first site of the active pair and contract
$$
\Theta_{auvc}=\sum_b A^{[i]}_{aub}A^{[i+1]}_{bvc}.
$$
The two physical indices enter the gate:
$$
\widetilde\Theta_{astc}=\sum_{u,v}G_{stuv}\Theta_{auvc},
\qquad G=e^{-ih_{i,i+1}\delta t}.
$$
The external indices $a,c$ still label orthonormal block states, so an SVD of $\widetilde\Theta_{(a,s),(t,c)}$ is a Schmidt decomposition across this bond.



Keep at most `chi` singular values. Their discarded squared sum is the lost norm at this update. We normalize the retained singular values and absorb them to the right, which puts the center at $i+1$. This is the same SVD placement used in a rightward DMRG update, with a local gate replacing the local eigenvalue solve.


In [5]:
def apply_two_site_gate(mps, bond, gate, chi):
    move_center(mps, bond)
    theta = np.einsum("asb,btc->astc", mps[bond], mps[bond + 1])
    theta = np.einsum("stuv,auvc->astc", gate.reshape(2, 2, 2, 2), theta)
    Dl, d1, d2, Dr = theta.shape
    U, singular, Vh = np.linalg.svd(theta.reshape(Dl * d1, d2 * Dr),
                                   full_matrices=False)
    keep = min(chi, len(singular))
    discarded = float(np.sum(singular[keep:]**2))
    singular = singular[:keep] / np.linalg.norm(singular[:keep])
    mps[bond] = U[:, :keep].reshape(Dl, d1, keep)
    mps[bond + 1] = (singular[:, None] * Vh[:keep]).reshape(keep, d2, Dr)
    return discarded

# Check one internal gate, including its site ordering, against dense action.
gate = gate_matrix(bonds[1], 0.037)
one_gate_mps = [A.copy() for A in mps_initial]
discarded = apply_two_site_gate(one_gate_mps, 1, gate, chi=4)
full_gate = np.kron(np.kron(I, gate), I)
assert np.allclose(mps_to_dense(one_gate_mps), full_gate @ psi_initial, atol=1e-12)
assert discarded < 1e-25
print("One-gate dense comparison error:",
      np.linalg.norm(mps_to_dense(one_gate_mps) - full_gate @ psi_initial))


One-gate dense comparison error: 1.484492924499981e-16


Separate the bonds into even and odd starting sites. Gates within either set act on disjoint pairs. A second-order step is
$$
U_2(\delta t)=e^{-iH_{\rm even}\delta t/2}
             e^{-iH_{\rm odd}\delta t}
             e^{-iH_{\rm even}\delta t/2}.
$$
The error of one untruncated step is $O(\delta t^3)$. At fixed total time, its accumulated state error is $O(\delta t^2)$. The code below implements the two half layers and the full middle layer in that order.


In [6]:
def tebd_step(mps, bonds, dt, chi):
    discarded = 0.0
    for parity, interval in [(0, dt / 2), (1, dt), (0, dt / 2)]:
        for bond in range(parity, len(bonds), 2):
            gate = gate_matrix(bonds[bond], interval)
            discarded += apply_two_site_gate(mps, bond, gate, chi)
    return discarded


def run_tebd(initial, bonds, dt, total_time, chi):
    steps = round(total_time / dt)
    assert abs(steps * dt - total_time) < 1e-12
    mps = [A.copy() for A in initial]
    discarded_sum = 0.0
    for _ in range(steps):
        discarded_sum += tebd_step(mps, bonds, dt, chi)
    return mps, discarded_sum


def phase_independent_error(state, reference):
    overlap = abs(np.vdot(reference, state))
    return np.sqrt(max(0.0, 2.0 - 2.0 * min(1.0, overlap)))

T = 0.6
psi_exact = unitary_action(H_dense, psi_initial, T)
tebd_results = {}
print("chi    dt       state error       energy change     sum discarded")
for chi in (2, 4):
    for dt in (0.1, 0.05, 0.025):
        mps, lost = run_tebd(mps_initial, bonds, dt, T, chi)
        state = mps_to_dense(mps)
        error = phase_independent_error(state, psi_exact)
        energy_change = float(np.vdot(state, H_dense @ state).real - E_initial)
        assert abs(np.linalg.norm(state) - 1) < 1e-11
        tebd_results[chi, dt] = {"state": state, "error": error, "discarded": lost}
        print(f"{chi:3d}  {dt:.3f}    {error:.6e}     {energy_change:+.6e}    {lost:.3e}")
ratio = tebd_results[4, 0.05]["error"] / tebd_results[4, 0.025]["error"]
assert 3.5 < ratio < 4.5
print(f"Untruncated error ratio on halving dt: {ratio:.4f}")


chi    dt       state error       energy change     sum discarded
  2  0.100    3.362180e-03     -8.360930e-04    1.667e-05
  2  0.050    4.233565e-03     -4.208045e-04    1.592e-05
  2  0.025    5.547369e-03     -2.382775e-04    1.119e-05
  4  0.100    1.743884e-03     -6.375809e-04    0.000e+00
  4  0.050    4.336643e-04     -1.619549e-04    0.000e+00
  4  0.025    1.082730e-04     -4.064605e-05    0.000e+00
Untruncated error ratio on halving dt: 4.0053


For four spins, $\chi=4$ can retain the full Schmidt rank. Its error ratio approaches four when the time step is halved, as expected for second-order evolution.

At $\chi=2$, reducing the time step also changes how often the state is truncated. The state error need not decrease monotonically because splitting and truncation errors can partially cancel. The sum of discarded weights records local norm losses at different updates. It is not the final infidelity.

## TDVP: project the Schrödinger velocity onto the MPS tangent space

Let $P_L^{[0:n)}$ project onto the Schmidt states of sites to the left of $n$, and let $P_R^{[n:N)}$ project onto the states on and to its right. For a fixed-rank MPS, the one-site tangent projector is
$$
P_T=\sum_{n=0}^{N-1}P_L^{[0:n)}\otimes I_n\otimes P_R^{[n+1:N)}
    -\sum_{b=1}^{N-1}P_L^{[0:b)}\otimes P_R^{[b:N)}.
$$
The first sum allows a variation of one center tensor. Neighboring center spaces overlap in a bond variation, so the second sum subtracts that overlap. The resulting equation is
$$
\frac{d}{dt}|\psi\rangle=-iP_T H|\psi\rangle.
$$
We will check this projector in the full ${16}$-dimensional Hilbert space before using its local factorization for time evolution.

For the check, write a cut's coefficient matrix as $M=U\Sigma V^\dagger$. The right *ket* vectors have coefficients `(Vh[:rank]).T`. Their projector is therefore `right @ right.conj().T`. Using `Vh.conj().T` here instead would conjugate the physical right states and fail for complex wavefunctions.


In [7]:
def tangent_projector(state, N, threshold=1e-12):
    left_projectors = [np.ones((1, 1), dtype=complex)]
    right_projectors = [None] * (N + 1)
    right_projectors[N] = np.ones((1, 1), dtype=complex)
    ranks = [1]
    for cut in range(1, N):
        U, singular, Vh = np.linalg.svd(state.reshape(2**cut, -1),
                                       full_matrices=False)
        rank = int(np.count_nonzero(singular > threshold))
        ranks.append(rank)
        left, right = U[:, :rank], Vh[:rank].T
        left_projectors.append(left @ left.conj().T)
        right_projectors[cut] = right @ right.conj().T
    ranks.append(1)
    projector = np.zeros((2**N, 2**N), dtype=complex)
    for site in range(N):
        projector += np.kron(np.kron(left_projectors[site], I),
                              right_projectors[site + 1])
    for cut in range(1, N):
        projector -= np.kron(left_projectors[cut], right_projectors[cut])
    return projector, ranks

P, ranks = tangent_projector(psi_initial, N)
assert np.allclose(P, P.conj().T, atol=1e-12)
assert np.allclose(P @ P, P, atol=1e-12)
assert np.allclose(P @ psi_initial, psi_initial, atol=1e-12)
tangent_dimension = sum(2 * ranks[i] * ranks[i + 1] for i in range(N))
tangent_dimension -= sum(rank**2 for rank in ranks[1:-1])
assert np.linalg.matrix_rank(P, tol=1e-10) == tangent_dimension
velocity = -1j * P @ H_dense @ psi_initial
assert abs(2 * np.vdot(psi_initial, velocity).real) < 1e-12
assert abs(2 * np.vdot(H_dense @ psi_initial, velocity).real) < 1e-12
print("Schmidt ranks:", ranks)
print("Tangent-space dimension:", tangent_dimension, "of", 2**N)
print("Discarded Schrödinger velocity ||(I-P)H psi||:",
      np.linalg.norm((np.eye(2**N) - P) @ H_dense @ psi_initial))


Schmidt ranks: [1, 2, 2, 2, 1]
Tangent-space dimension: 12 of 16
Discarded Schrödinger velocity ||(I-P)H psi||: 0.0036233148377265915


The two scalar checks in the last cell give the instantaneous changes of norm and energy. Both vanish because $P_T$ is Hermitian and $P_T|\psi\rangle=|\psi\rangle$.

The projector removes some Schrödinger velocities when the available ranks are restricted. Norm and energy conservation can therefore coexist with a nonzero error in the evolved state. We will see this distinction in the final comparison.

## Build the local TDVP actions from MPO environments

Use the Ising MPO and the bra-first environment convention from the DMRG derivation. The recursions are repeated here so the notebook has no dependency on another file:
$$
L'_{uv\nu}=\sum_{a,b,\mu,s,t}\overline{A_{asu}}L_{ab\mu}W_{\mu\nu st}A_{btv},
\quad
R'_{ab\mu}=\sum_{u,v,\nu,s,t}\overline{A_{asu}}W_{\mu\nu st}A_{btv}R_{uv\nu}.
$$
Each environment has axes `(bra bond, ket bond, MPO bond)`. The boundary environment is `(1,1,1)` with value one.


In [8]:
def ising_mpo(N, J, hx, hz):
    W = np.zeros((3, 3, 2, 2), dtype=complex)
    W[0, 0], W[1, 0] = I, Z
    W[2, 0], W[2, 1], W[2, 2] = -hx * X - hz * Z, -J * Z, I
    return [W[2:3].copy()] + [W.copy() for _ in range(N - 2)] + [W[:, 0:1].copy()]


def absorb_left(L, A, W):
    return np.einsum("asu,abm,mnst,btv->uvn", A.conj(), L, W, A,
                     optimize=True)


def absorb_right(R, A, W):
    return np.einsum("asu,mnst,btv,uvn->abm", A.conj(), W, A, R,
                     optimize=True)


def right_environments(mps, mpo):
    right = [None] * (len(mps) + 1)
    right[-1] = np.ones((1, 1, 1), dtype=complex)
    for i in range(len(mps) - 1, -1, -1):
        right[i] = absorb_right(right[i + 1], mps[i], mpo[i])
    return right

mpo = ising_mpo(N, J, hx, hz)
right = right_environments(mps_initial, mpo)
assert abs(right[0].item() - E_initial) < 1e-11
print("Initial energy from MPO environments:", right[0].item())


Initial energy from MPO environments: (-4.445416067263645-1.856901481383301e-20j)


At site $n$, the center tensor has shape $(D_n,2,D_{n+1})$. Its effective Hamiltonian is
$$
(H_n)_{asc,\,btd}=\sum_{\mu,\nu}L_{ab\mu}W^{[n]}_{\mu\nu st}R_{cd\nu}.
$$
After a QR factorization, the center becomes a bond matrix $C_{bd}$. Its effective Hamiltonian uses no physical site:
$$
(K_n)_{ac,\,bd}=\sum_\mu L_{ab\mu}R_{cd\mu}.
$$
These are the one-site and zero-site versions of the DMRG contraction. The diagrams retain open output legs and sum only the legs attached to the trial tensor.



The positive terms of the tangent projector give center evolution $A_C\mapsto e^{-iH_n\delta t}A_C$. Its negative bond terms give $C\mapsto e^{+iK_n\delta t}C$. The backward bond step is essential. Omitting it would count shared variations more than once.


In [9]:
def center_hamiltonian(L, W, R):
    tensor = np.einsum("abm,mnst,cdn->ascbtd", L, W, R, optimize=True)
    size = L.shape[0] * W.shape[2] * R.shape[0]
    matrix = tensor.reshape(size, size)
    assert np.linalg.norm(matrix - matrix.conj().T) < 1e-10
    return matrix


def bond_hamiltonian(L, R):
    tensor = np.einsum("abm,cdm->acbd", L, R, optimize=True)
    size = L.shape[0] * R.shape[0]
    matrix = tensor.reshape(size, size)
    assert np.linalg.norm(matrix - matrix.conj().T) < 1e-10
    return matrix

left_boundary = np.ones((1, 1, 1), dtype=complex)
H_center = center_hamiltonian(left_boundary, mpo[0], right[1])
A_left, C = factor_left(mps_initial[0])
L_after = absorb_left(left_boundary, A_left, mpo[0])
K_bond = bond_hamiltonian(L_after, right[1])
print("First center Hamiltonian shape:", H_center.shape)
print("First bond Hamiltonian shape:", K_bond.shape)


First center Hamiltonian shape: (4, 4)
First bond Hamiltonian shape: (4, 4)


## A complete symmetric one-site TDVP step

Start with the center on site zero and all later sites right-canonical.

1. On each site except the last, evolve the center forward by $\delta t/2$.
2. Factor it as $A_L C$, absorb $A_L$ into the left environment, evolve $C$ backward by $\delta t/2$, and absorb it into the next site.
3. Evolve the last center forward by the full $\delta t$.
4. Move left by factoring the current center as $C A_R$. Update the right environment, evolve the bond backward by $\delta t/2$, and absorb it into the previous site.
5. Evolve that center forward by $\delta t/2$ and repeat until reaching site zero.

The last site's full step joins the two half steps at the turn of the sweep. The second half reverses the order of the local operations, which makes the composition symmetric. The left environments saved during the first half remain valid during the return sweep because the tensors to the left of the center have not yet changed.

The code uses exact dense exponentials of these small local matrices. It performs no SVD truncation and leaves every virtual dimension unchanged.


In [10]:
def tdvp_step(mps, mpo, dt):
    right = right_environments(mps, mpo)
    left = [np.ones((1, 1, 1), dtype=complex)]
    # Half sweep to the right: center forward, bond backward.
    for i in range(len(mps) - 1):
        H = center_hamiltonian(left[i], mpo[i], right[i + 1])
        mps[i] = unitary_action(H, mps[i], dt / 2)
        mps[i], C = factor_left(mps[i])
        left.append(absorb_left(left[i], mps[i], mpo[i]))
        K = bond_hamiltonian(left[i + 1], right[i + 1])
        C = unitary_action(K, C, -dt / 2)
        mps[i + 1] = np.einsum("ab,bsd->asd", C, mps[i + 1])
    # Join the two half steps at the last site.
    H = center_hamiltonian(left[-1], mpo[-1], right[-1])
    mps[-1] = unitary_action(H, mps[-1], dt)
    # Half sweep to the left: bond backward, center forward.
    R = right[-1]
    for i in range(len(mps) - 1, 0, -1):
        C, mps[i] = factor_right(mps[i])
        R = absorb_right(R, mps[i], mpo[i])
        K = bond_hamiltonian(left[i], R)
        C = unitary_action(K, C, -dt / 2)
        mps[i - 1] = np.einsum("asb,bc->asc", mps[i - 1], C)
        H = center_hamiltonian(left[i - 1], mpo[i - 1], R)
        mps[i - 1] = unitary_action(H, mps[i - 1], dt / 2)


We can test the sign of the backward bond step without comparing two implementations of the same sweep. Evaluate the finite difference of the complete MPS step and compare it with the independently assembled tangent velocity:
$$
\frac{|\psi(\epsilon)\rangle-|\psi(-\epsilon)\rangle}{2\epsilon}
\simeq -iP_T H|\psi(0)\rangle.
$$
We also reverse a finite step. Symmetry of the composition should return the initial physical state, including its phase, up to numerical roundoff.


In [11]:
epsilon = 1e-5
plus = [A.copy() for A in mps_initial]
minus = [A.copy() for A in mps_initial]
tdvp_step(plus, mpo, epsilon)
tdvp_step(minus, mpo, -epsilon)
finite_difference = (mps_to_dense(plus) - mps_to_dense(minus)) / (2 * epsilon)
velocity_error = np.linalg.norm(finite_difference - velocity)
assert velocity_error < 1e-7

reversed_state = [A.copy() for A in mps_initial]
tdvp_step(reversed_state, mpo, 0.1)
tdvp_step(reversed_state, mpo, -0.1)
reverse_error = np.linalg.norm(mps_to_dense(reversed_state) - psi_initial)
assert reverse_error < 1e-10
print(f"TDVP velocity comparison error: {velocity_error:.3e}")
print(f"Forward-then-backward state error: {reverse_error:.3e}")


TDVP velocity comparison error: 2.169e-09
Forward-then-backward state error: 1.315e-15


## Fixed-rank evolution and conservation

During the evolution, record the norm and the expectation of the time-independent Hamiltonian. Each local center or bond substep is an exact unitary evolution in its current orthonormal block basis. Its local Hamiltonian is the restriction of the full $H$, so these substeps conserve the global energy as well as the norm, up to numerical roundoff.

A small time step improves the integration of the fixed-rank TDVP equation. It does not restore the part of $H|\psi\rangle$ removed by the tangent projector. The comparison below uses the same initial state and total time as TEBD.


In [12]:
def run_tdvp(initial, mpo, dt, total_time):
    steps = round(total_time / dt)
    assert abs(steps * dt - total_time) < 1e-12
    mps = [A.copy() for A in initial]
    original_shapes = [A.shape for A in mps]
    energies, norms = [], []
    for _ in range(steps):
        tdvp_step(mps, mpo, dt)
        state = mps_to_dense(mps)
        norms.append(np.linalg.norm(state))
        energies.append(float(np.vdot(state, H_dense @ state).real))
        assert [A.shape for A in mps] == original_shapes
    return mps, np.array(energies), np.array(norms)

tdvp_results = {}
print("dt       state error       max energy drift    max norm drift")
for dt in (0.1, 0.05, 0.025):
    mps, energies, norms = run_tdvp(mps_initial, mpo, dt, T)
    state = mps_to_dense(mps)
    error = phase_independent_error(state, psi_exact)
    energy_drift = np.max(np.abs(energies - E_initial))
    norm_drift = np.max(np.abs(norms - 1))
    assert energy_drift < 1e-10
    assert norm_drift < 1e-11
    tdvp_results[dt] = {"state": state, "error": error}
    print(f"{dt:.3f}    {error:.6e}     {energy_drift:.3e}         {norm_drift:.3e}")


dt       state error       max energy drift    max norm drift
0.100    6.634512e-03     9.770e-14         1.021e-14
0.050    7.063926e-03     6.928e-14         7.216e-15
0.025    7.131436e-03     1.510e-13         1.554e-14


The TDVP state error approaches a nonzero value as the time step decreases because these runs keep bond dimension two. The energy remains conserved, while untruncated TEBD approaches the exact unitary result as its time step decreases.

To separate integration error from rank restriction, compare the TDVP runs with a finer TDVP run on the same manifold. We compare physical state vectors after removing an overall phase. The final spin expectation gives a concrete observable comparison alongside the state error.


In [13]:
reference_mps, _, _ = run_tdvp(mps_initial, mpo, 0.00625, T)
reference_tdvp = mps_to_dense(reference_mps)
coarse_error = phase_independent_error(tdvp_results[0.1]["state"], reference_tdvp)
fine_error = phase_independent_error(tdvp_results[0.025]["state"], reference_tdvp)
assert fine_error < coarse_error
print(f"TDVP error relative to fine fixed-rank run: {coarse_error:.3e} -> {fine_error:.3e}")

Z_center = operator_on_sites(N, {N // 2: Z})
for name, state in [("exact", psi_exact),
                    ("TEBD chi=4, dt=.025", tebd_results[4, 0.025]["state"]),
                    ("TDVP chi=2, dt=.025", tdvp_results[0.025]["state"])]:
    spin = np.vdot(state, Z_center @ state).real
    print(f"{name:24s} <Z_center> = {spin:.10f}")
print(f"Notebook calculation time: {time.perf_counter() - started:.3f} s")


TDVP error relative to fine fixed-rank run: 9.797e-04 -> 4.146e-05
exact                    <Z_center> = 0.5586695449
TEBD chi=4, dt=.025      <Z_center> = 0.5586977577
TDVP chi=2, dt=.025      <Z_center> = 0.5588302433
Notebook calculation time: 1.044 s


## Exercises

1. **Gate and truncation errors.** Repeat the TEBD table at $T=1.2$ with $\chi=2$ and ${4}$. Compare the final state error and the sum of discarded weights. For the untruncated runs, check whether halving $\delta t$ still reduces the state error by approximately four. Explain any change in the restricted-rank trend without identifying the sum of discarded weights with final infidelity.

2. **A product-state TDVP manifold.** Set the initial MPS to $|+\rangle^{\otimes4}$ and evolve under $H=-\sum_iZ_iZ_{i+1}$, with both fields zero. Compare one-site TDVP at bond dimension one with TEBD allowed to grow to bond dimension four. Compute the dense tangent projector and its discarded velocity. One-site TDVP preserves a product state, whereas the exact Ising evolution develops correlations. The cell below checks the initial projected velocity for this case.

3. **Give TDVP more Schmidt states.** Construct a full-rank initial MPS from `initial_dense` with `chi=4`, apply the same local phase rotation, and repeat the quench. Recompute the exact reference for this new initial state. Check that its tangent projector is the identity on the full Hilbert space. Compare time-step errors with the fixed-rank-two results. Do not append zero Schmidt values to the rank-two state and treat them as populated variational directions.


In [14]:
# A completed check for the product-state exercise.
plus_spin = np.array([1, 1], dtype=complex) / np.sqrt(2)
product = [plus_spin.reshape(1, 2, 1).copy() for _ in range(N)]
product_vector = mps_to_dense(product)
product_P, product_ranks = tangent_projector(product_vector, N)
H_zz = dense_ising(N, 1.0, 0.0, 0.0)
projected = product_P @ H_zz @ product_vector
assert np.linalg.norm(projected) < 1e-12
assert np.linalg.norm(H_zz @ product_vector) > 1
print("Product-state Schmidt ranks:", product_ranks)
print("||P H_ZZ |++++>||:", np.linalg.norm(projected))
print("||H_ZZ |++++>||:", np.linalg.norm(H_zz @ product_vector))
print("All TEBD, tangent-projector, TDVP sweep, and conservation checks passed.")


Product-state Schmidt ranks: [1, 1, 1, 1, 1]
||P H_ZZ |++++>||: 3.669093936010194e-16
||H_ZZ |++++>||: 1.7320508075688767
All TEBD, tangent-projector, TDVP sweep, and conservation checks passed.
